# fewshot-msme-inspect - Kaggle setup

Mirrors the local CPU environment so a free Kaggle GPU burst (P100 or T4 x2,
30 h/week) can run the heavy bits - feature bank builds, adapter training
sweeps - while every reported latency / RAM / FLOP number still comes from the
local CPU benchmark.

**Before running anything, in the right-hand panel:**

1. **Session options -> Accelerator -> GPU P100** (or T4 x2)
2. **Session options -> Internet -> On** (needs a phone-verified account; without
   it `pip install` and `git clone` both fail)
3. **Add-ons -> Secrets -> add `GITHUB_TOKEN`** and attach it to this notebook -
   the repo is private, so the clone needs it
4. **Add Data -> search "MVTec AD"** and attach it, so the dataset is mounted
   read-only at `/kaggle/input` instead of being downloaded

Then run the cells top to bottom.


## 1. What this session got

In [ ]:
import os, platform, subprocess, sys

print("python", platform.python_version())

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print("gpu   ", gpu or "none reported")
except FileNotFoundError:
    print("gpu    no nvidia-smi - this is a CPU session")

ON_KAGGLE = os.path.isdir("/kaggle")
print("kaggle ", ON_KAGGLE)


## 2. Kaggle's directory layout

Three directories matter, and they behave differently:

| path | writable | survives the session |
|---|---|---|
| `/kaggle/input` | no | yes, it is the attached datasets |
| `/kaggle/working` | yes, 20 GB | only if you **Save Version** |
| `/kaggle/temp` | yes | no, wiped at session end |

Anything worth keeping - feature banks, adapter checkpoints - has to be in
`/kaggle/working` **and** you have to save a version, or it is gone.


In [ ]:
WORKING = "/kaggle/working" if ON_KAGGLE else os.getcwd()
TEMP = "/kaggle/temp" if ON_KAGGLE else os.path.join(os.getcwd(), "tmp")
INPUT = "/kaggle/input"

os.makedirs(TEMP, exist_ok=True)
print("working:", WORKING)
print("temp   :", TEMP)

if os.path.isdir(INPUT):
    attached = sorted(os.listdir(INPUT))
    print("attached datasets:", attached if attached else "(none - use Add Data)")


## 3. Get the repo

Private repo, so the clone needs a token. Add a fine-grained GitHub PAT with
read access to `fewshot-msme-inspect` under **Add-ons -> Secrets** as
`GITHUB_TOKEN`. The token is used for the clone and then dropped from the
remote, so it is not left sitting in `.git/config`.

Re-running this cell pulls instead of re-cloning.


In [ ]:
REPO_URL = "https://github.com/Mounika-Reddy-0802/fewshot-msme-inspect.git"
REPO_DIR = os.path.join(WORKING, "fewshot-msme-inspect")

TOKEN = None
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as exc:
        print("no GITHUB_TOKEN secret:", type(exc).__name__)
if not TOKEN:
    from getpass import getpass
    TOKEN = getpass("GitHub token (blank if the repo is public): ").strip() or None

clone_url = REPO_URL.replace("https://", "https://" + TOKEN + "@") if TOKEN else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone --quiet {clone_url} {REPO_DIR}
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


## 4. Install

Kaggle ships torch already, and it is the CUDA build - leave it alone. Only the
packages Kaggle is missing get installed. None of them declare a torch
dependency, so the GPU build survives; the cell after checks that it did.


In [ ]:
import torch
print("preinstalled torch:", torch.__version__, "cuda:", torch.cuda.is_available())

!pip install -q open_clip_torch==3.3.0 anomalib==2.6.0 fvcore thop


In [ ]:
# sanity: nothing above swapped the CUDA build for a CPU one
import open_clip, torch

print("torch    ", torch.__version__, "cuda available:", torch.cuda.is_available())
print("open_clip", open_clip.__version__)
if ON_KAGGLE and not torch.cuda.is_available():
    print("WARNING: no GPU - Session options -> Accelerator -> GPU")


## 5. Point the roots at Kaggle storage

Same environment variables as the local setup, so no config or script changes
between machines.


In [ ]:
os.environ["FEWSHOT_DATA_ROOT"] = os.path.join(TEMP, "data")
os.environ["FEWSHOT_RESULTS_ROOT"] = os.path.join(WORKING, "results")
os.makedirs(os.environ["FEWSHOT_DATA_ROOT"], exist_ok=True)
os.makedirs(os.environ["FEWSHOT_RESULTS_ROOT"], exist_ok=True)

from src.utils.paths import data_root, results_root
print("data   ", data_root(), "(scratch - anything downloaded here is disposable)")
print("results", results_root(), "(Save Version to keep)")


## 6. Find MVTec AD

If you attached an MVTec dataset with **Add Data**, it is already on disk at
`/kaggle/input` and nothing needs downloading - which also means it does not eat
the session's time budget.

This looks for a directory that has several MVTec category folders side by side,
whatever the dataset happens to be called, so it does not depend on one specific
Kaggle dataset slug. If none is attached it falls back to pulling a single
category from the mirror.


In [ ]:
from src.utils.paths import MVTEC_CATEGORIES, find_category_root

# find_category_root lives in src/utils/paths.py so it is covered by the test
# suite rather than being untested notebook-only code
found_root = find_category_root(INPUT)

if found_root:
    MVTEC_ROOT = str(found_root)
    present = sorted(set(MVTEC_CATEGORIES) & set(os.listdir(MVTEC_ROOT)))
    print("found attached MVTec at:", MVTEC_ROOT)
    print("categories:", len(present), present)
else:
    print("no MVTec dataset attached - fetching one category from the mirror")
    !python scripts/download_mvtec_category.py --category bottle
    MVTEC_ROOT = os.path.join(os.environ["FEWSHOT_DATA_ROOT"], "mvtec_ad")
    print("using:", MVTEC_ROOT)


## 7. Configs resolve here too

`dataset.root` is overridden rather than edited, because the attached dataset
sits wherever Kaggle mounted it. The override goes through the same dotted-key
mechanism the command line uses, so nothing about the config files changes.


In [ ]:
from src.utils.config import load_config
from src.utils.seed import seed_everything

config = load_config("prototype.yaml", overrides=["dataset.root=" + MVTEC_ROOT])
seed_everything(config["experiment"]["seed"])

print("experiment:", config["experiment"])
print("backbone:  ", config["backbone"])
print("dataset:   ", config["dataset"]["name"], "at", config["dataset"]["root"])


## 8. Smoke test: encode with the frozen backbone

Downloads the ViT-B/16 weights once and runs one image and one text batch
through them. Two feature shapes and a similarity matrix means this environment
matches the local one.


In [ ]:
import open_clip, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
name, pretrained = config["backbone"]["name"], config["backbone"]["pretrained"]

model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=pretrained)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(name)

images = torch.zeros(2, 3, 224, 224, device=device)
text = tokenizer(["a photo of a bottle", "a photo of a bottle with a defect"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(images)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    similarity = image_features @ text_features.T

print("device       ", device)
print("backbone     ", name, "/", pretrained)
print("image feats  ", tuple(image_features.shape))
print("text feats   ", tuple(text_features.shape))
print("similarity   ", similarity.cpu().numpy().round(4))


## 9. Record what this session was

Kaggle hands out P100 or T4 depending on availability. Anything produced here
gets stamped with the session it came from, so results stay traceable.


In [ ]:
import json, subprocess

runtime = {
    "platform": "kaggle" if ON_KAGGLE else "local",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": device,
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() or "cpu",
    "open_clip": open_clip.__version__,
    "backbone": name + "/" + pretrained,
    "mvtec_root": MVTEC_ROOT,
}
print(json.dumps(runtime, indent=2))

out = os.path.join(os.environ["FEWSHOT_RESULTS_ROOT"], "kaggle_runtime.json")
with open(out, "w") as f:
    json.dump(runtime, f, indent=2)
print("written to", out)


---

## Before you close the tab

* **Save Version** (top right) or everything in `/kaggle/working` is discarded.
  For anything big - feature banks, checkpoints - save it as a Kaggle Dataset
  output instead and attach it to the next session.
* A session is capped at **12 hours**, and the GPU quota is **30 h/week** on a
  rolling reset. Long adapter sweeps should checkpoint so they can resume.
* **Latency, RAM and FLOPs for the paper are measured locally on CPU.** The GPU
  here is only for building feature banks and training adapter sweeps faster -
  never for a number that ends up in the efficiency benchmark.
